In [3]:
from pyspark.sql import SparkSession
from pyspark. sql. functions import col, desc, sum, avg

spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 04:51:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
products = spark.read.csv("products.csv", header=True, inferSchema=True)
orders = spark.read.csv("orders.csv", header=True, inferSchema=True)
order_items = spark.read.csv("order_items.csv", header=True, inferSchema=True)
customers = spark.read.csv("customers.csv", header=True, inferSchema=True)
returns = spark.read.csv("returns.csv", header=True, inferSchema=True)

26/06/17 04:51:42 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [5]:
products.createOrReplaceTempView("products")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
customers.createOrReplaceTempView("customers")
returns.createOrReplaceTempView("returns")

In [6]:
spark.sql("SHOW TABLES").show()

+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|         |  customers|       true|
|         |order_items|       true|
|         |     orders|       true|
|         |   products|       true|
|         |    returns|       true|
+---------+-----------+-----------+



In [7]:
import os
if not os.path.exists("output"):
    os.makedirs("output")

In [8]:
q1_customers = spark.sql("""SELECT COUNT(*) AS total_customers FROM customers""")

In [9]:
q1_customers.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q1_customers")

In [10]:
q1_products = spark.sql("""SELECT COUNT(*) AS total_products FROM products""")

In [11]:
q1_products.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q1_products")

In [12]:
q1_orders = spark.sql("""SELECT COUNT(*) AS orders FROM orders""")

In [13]:
q1_orders.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q1_orders")

In [14]:
q1_returns = spark.sql("""SELECT COUNT(*) AS total_returned_orders FROM returns""")

In [15]:
q1_returns.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q1_returns")

In [16]:
#total sales amount  generated by each product category 

In [17]:
q2 = spark.sql("""SELECT
    p.category,
    SUM(oi.quantity * oi.selling_price) AS total_sales
FROM order_items oi
JOIN products p
ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_sales DESC""")

In [18]:
q2.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q2")

In [19]:
#top 10 customers based on total purchase amount

In [20]:
customers.printSchema()
orders.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- order_status: string (nullable = true)



In [21]:
q3 = spark.sql("""SELECT
    c.customer_id,
    c.customer_name,
    SUM(oi.quantity * oi.selling_price) AS total_purchase
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY total_purchase DESC
LIMIT 10""")

In [22]:
q3.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q3")

In [23]:
#calculate monthly sales  trends for the last available year

In [24]:
orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- order_status: string (nullable = true)



In [25]:
q4 = spark.sql("""SELECT
    MONTH(o.order_date) AS month,
    SUM(oi.quantity * oi.selling_price) AS monthly_sales
FROM orders o
JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE YEAR(o.order_date) = (
    SELECT MAX(YEAR(order_date))
    FROM orders
)
GROUP BY MONTH(o.order_date)
ORDER BY month""")

In [26]:
q4.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q4")

In [27]:
#Return percentage for each product category

In [28]:
q5 = spark.sql("""SELECT
    p.category,
    ROUND(
        COUNT(DISTINCT r.order_id) * 100.0 /
        COUNT(DISTINCT o.order_id),
        2
    ) AS return_percentage
FROM products p
JOIN order_items oi
ON p.product_id = oi.product_id
JOIN orders o
ON oi.order_id = o.order_id
LEFT JOIN returns r
ON o.order_id = r.order_id
GROUP BY p.category""")

In [29]:
q5.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q5")

In [30]:
#the most preferred payment mode in each state 

In [31]:
q6 = spark.sql("""WITH payment_counts AS (
    SELECT
        c.state,
        o.payment_mode,
        COUNT(*) AS total_orders,
        ROW_NUMBER() OVER (
            PARTITION BY c.state
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.state, o.payment_mode
)

SELECT
    state,
    payment_mode,
    total_orders
FROM payment_counts
WHERE rn = 1""")


In [32]:
q6.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q6")

In [33]:
#identify customers  who have purchased products from at least 5 different  categories and spent more than 1,00,000.

In [34]:
q7 = spark.sql("""SELECT
    c.customer_id,
    c.customer_name,
    COUNT(DISTINCT p.category) AS category_count,
    SUM(oi.quantity * oi.selling_price) AS total_spent
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY
    c.customer_id,
    c.customer_name
HAVING
    COUNT(DISTINCT p.category) >= 5
    AND SUM(oi.quantity * oi.selling_price) > 100000
ORDER BY total_spent DESC""")

In [35]:
q7.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q7")

26/06/17 04:52:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/17 04:52:45 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

In [36]:
#top 3 products  by revenue within each category

In [37]:
#revenue = quantity*amount

In [38]:
q8 = spark.sql("""WITH product_revenue AS (
    SELECT
        p.category,
        p.product_id,
        p.product_name,
        SUM(oi.quantity * oi.selling_price) AS revenue,
        ROW_NUMBER() OVER (
            PARTITION BY p.category
            ORDER BY SUM(oi.quantity * oi.selling_price) DESC
        ) AS rank
    FROM products p
    JOIN order_items oi
        ON p.product_id = oi.product_id
    GROUP BY
        p.category,
        p.product_id,
        p.product_name
)

SELECT
    category,
    product_id,
    product_name,
    revenue
FROM product_revenue
WHERE rank <= 3
ORDER BY category, revenue DESC""")

In [39]:
q8.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q8")

In [40]:
#customer lifetime value (CLV) and classify customers into Bronze, Silver, and Gold

In [44]:
q9 = spark.sql("""SELECT
    c.customer_id,
    c.customer_name,
    SUM(oi.quantity * oi.selling_price) AS clv,

    CASE
        WHEN SUM(oi.quantity * oi.selling_price) < 50000
            THEN 'Bronze'

        WHEN SUM(oi.quantity * oi.selling_price) BETWEEN 50000 AND 100000
            THEN 'Silver'

        ELSE 'Gold'
    END AS customer_segment

FROM customers c

JOIN orders o
    ON c.customer_id = o.customer_id

JOIN order_items oi
    ON o.order_id = oi.order_id

GROUP BY
    c.customer_id,
    c.customer_name

ORDER BY clv DESC""")

In [45]:
q9.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q9")

In [46]:
#10 Create a final business analytics dataset containing customer, order, revenue, return, and
 #segmentation information and export it in Parquet format partitioned by state

In [47]:
q10 = spark.sql("""SELECT
    c.customer_id,
    c.customer_name,
    c.city,
    c.state,

    o.order_id,
    o.order_date,
    o.payment_mode,
    o.order_status,

    p.category,

    oi.quantity,
    oi.selling_price,

    (oi.quantity * oi.selling_price) AS revenue,

    CASE
        WHEN r.order_id IS NOT NULL
        THEN 'Returned'
        ELSE 'Not Returned'
    END AS return_status,

    CASE
        WHEN cust_clv.clv < 50000
            THEN 'Bronze'

        WHEN cust_clv.clv BETWEEN 50000 AND 100000
            THEN 'Silver'

        ELSE 'Gold'
    END AS customer_segment

FROM customers c

JOIN orders o
    ON c.customer_id = o.customer_id

JOIN order_items oi
    ON o.order_id = oi.order_id

JOIN products p
    ON oi.product_id = p.product_id

LEFT JOIN returns r
    ON o.order_id = r.order_id

JOIN (
    SELECT
        c.customer_id,
        SUM(oi.quantity * oi.selling_price) AS clv
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    JOIN order_items oi
        ON o.order_id = oi.order_id
    GROUP BY c.customer_id
) cust_clv
ON c.customer_id = cust_clv.customer_id""")

In [48]:
q10.show(5)

+-----------+--------------+---------+-----+--------+----------+----------------+------------+-----------+--------+-------------+-----------------+-------------+----------------+
|customer_id| customer_name|     city|state|order_id|order_date|    payment_mode|order_status|   category|quantity|selling_price|          revenue|return_status|customer_segment|
+-----------+--------------+---------+-----+--------+----------+----------------+------------+-----------+--------+-------------+-----------------+-------------+----------------+
|      63271|Customer_63271|    Miami|   FL|     854|2024-08-14|Cash on Delivery|    Returned|     Sports|       1|       932.02|           932.02|     Returned|          Silver|
|      63271|Customer_63271|    Miami|   FL|     854|2024-08-14|Cash on Delivery|    Returned|   Clothing|       2|       617.76|          1235.52|     Returned|          Silver|
|      63271|Customer_63271|    Miami|   FL|     854|2024-08-14|Cash on Delivery|    Returned|     Beauty

In [49]:
q10.coalesce(1).write.mode("overwrite").option("header", True).csv("output/q10")